In [1]:
import copy
import salvus.namespace as sn
import numpy as np
import matplotlib.pyplot as plt

import sys
import importlib
from pathlib import Path
from scipy.interpolate import interp1d

path_to_add = str(Path.cwd() / "tomography")
if path_to_add not in sys.path:
    sys.path.append(path_to_add)

import my_code.utilities
importlib.reload(my_code.utilities)
from my_code.utilities import *
import salvus

import salvus.mesh.layered_meshing as lm
from datetime import datetime
import salvus.flow.simple_config as sc
from salvus.toolbox.helpers.wavefield_output import (
    WavefieldOutput,
    wavefield_output_to_xarray,
)
import xarray as xr


--> Server: 'https://l.mondaic.com/licensing_server', User: 'bristol.support', Group: 'UniversityOfBristol'.
--> Negotiating 1 license instance(s) for 'SalvusMesh' [license version 1.0.0] for 1 seconds ...
--> Success! [Total duration: 0.14 seconds]


## Configuration

In [2]:
# ---- Salvus site --------------------------------------------------------
SITE_NAME = "isambard_oliver"
RANKS     = 8

# ---- Physical parameters ------------------------------------------------
VP        = 1500.0
RHO       = 1025.0
THICKNESS = 0.2    # m — layer thickness (used to interpret alpha)
vc        = 1400   # VP inside the layer [m/s]

# ---- Multi-scale frequency schedule -------------------------------------
alpha_list           = [0.2, 0.4, 0.6]   # thickness-to-wavelength ratios (low → high)
MAX_ITER_PER_SCALE   = 5
SAMPLING_INTERVAL    = 10

# ---- Domain -------------------------------------------------------------
x0_dom, x1_dom = 0.0, 1.0
y0_dom, y1_dom = 0.0, 1.0
domain = sn.domain.dim2.BoxDomain(x0=x0_dom, x1=x1_dom, y0=y0_dom, y1=y1_dom)

# ---- Project / data directories -----------------------------------------
PROJECT_DIR = '/home/b6as/oliverwfy.b6as/workspace/acoustic_model/Project'
DATA_DIR    = Path('/home/b6as/oliverwfy.b6as/workspace/acoustic_model/data')
IMAGE_DIR   = Path('/home/b6as/oliverwfy.b6as/workspace/acoustic_FWI_velocity_underwater/image')

Path(PROJECT_DIR).mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# ---- Mesh parameters (same across all scales) ---------------------------
elements_per_wavelength = 3
model_order             = 4
number_of_wavelengths   = 2
free_surfaces           = ['y0', 'y1']

# ---- End time per scale ------------------------------------------------
end_time_per_alpha = {0.2: 1.2e-3, 0.4: 0.8e-3, 0.6: 0.6e-3}

# ---- ROI ----------------------------------------------------------------
ROI = (0.0, 1.0, 0.4, 0.6)   # (x0, x1, y0, y1)

# ---- True (layered) model -----------------------------------------------
m_true = sn.layered_meshing.LayeredModel([
    sn.material.from_params(rho=RHO, vp=VP),
    sn.layered_meshing.interface.Hyperplane.at(0.6),
    sn.material.from_params(rho=RHO, vp=vc),
    sn.layered_meshing.interface.Hyperplane.at(0.4),
    sn.material.from_params(rho=RHO, vp=VP),
])


## Source / Receiver Setup

In [ ]:
event_name      = 'single_pairs'
amplitude_ratio = 6e4

s_loc = (0.5, 0.3)
r_loc = [(0.5, 0.7)]

receivers = [
    sn.simple_config.receiver.cartesian.Point2D(
        x=r[0], y=r[1], station_code=f"R{j+1}", fields=["phi"]
    )
    for j, r in enumerate(r_loc)
]
source = sn.simple_config.source.cartesian.ScalarPoint2D(
    x=s_loc[0], y=s_loc[1], f=amplitude_ratio
)


## Helper Functions

In [ ]:
def forward_simulation(simulation_name, events,
                       fields=None, sampling_interval_in_time_steps=10):
    if fields:
        p.simulations.launch(
            ranks_per_job=RANKS,
            site_name=SITE_NAME,
            events=events,
            simulation_configuration=simulation_name,
            extra_output_configuration={
                "volume_data": {
                    "sampling_interval_in_time_steps": sampling_interval_in_time_steps,
                    "fields": fields,
                },
            },
            delete_conflicting_previous_results=True,
        )
    else:
        p.simulations.launch(
            ranks_per_job=RANKS,
            site_name=SITE_NAME,
            events=events,
            simulation_configuration=simulation_name,
            delete_conflicting_previous_results=True,
        )
    p.simulations.query(block=True, verbosity=1)


def extract_data(simulation, events, receiver_field='phi', field=None):
    ed = p.waveforms.get(data_name=simulation, events=events)[0]
    data = ed.get_waveform_data_xarray(receiver_field=receiver_field)
    wavefield = None
    if field:
        wavefield = wavefield_output_to_xarray(
            ed.get_wavefield_output(output_type='volume', field=field),
            p.simulations.get_mesh(simulation).points,
        ).T
    return data, wavefield


def misfit_adjoint_source(u_obs, u, event=False, iteration=0):
    sampling_rate = u_obs.sampling_rate_in_hertz
    start_time    = u_obs.time[0]
    end_time      = u_obs.time[-1]

    diff   = u_obs - u
    misfit = 0.5 / sampling_rate * (diff ** 2).sum()
    f_adj  = np.flip(diff, axis=1) / sampling_rate

    event_adj        = None
    event_config_adj = None

    if event:
        rxs, srcs = p.events.get(event).receivers, p.events.get(event).sources
        rxs_adj_loc = [r.location for r in srcs]
        src_adj_loc = [t.location for t in rxs]

        stf_adj_ls = [
            sc.stf.Custom.from_array(
                np.array(f_adj[i]),
                sampling_rate_in_hertz=sampling_rate_in_hertz,
                start_time_in_seconds=0.0,
            )
            for i in range(len(src_adj_loc))
        ]
        srcs_adj = [
            sn.simple_config.source.cartesian.ScalarPoint2D(x=tx[0], y=tx[1], f=1)
            for tx in src_adj_loc
        ]
        rxs_adj = [
            sn.simple_config.receiver.cartesian.Point2D(
                x=rx[0], y=rx[1],
                fields=['phi'],
                station_code=f"ite:{iteration}_adj_{_i:06d}",
            )
            for _i, rx in enumerate(rxs_adj_loc)
        ]
        event_adj = f'{event}_adjoint_{iteration}'
        p.add_to_project(sn.Event(event_name=event_adj, sources=srcs_adj, receivers=rxs_adj))
        event_config_adj = sn.EventConfiguration(
            wavelet=stf_adj_ls,
            waveform_simulation_configuration=sn.WaveformSimulationConfiguration(
                start_time_in_seconds=start_time,
                end_time_in_seconds=end_time,
                time_step_in_seconds=1 / sampling_rate_in_hertz,
            ),
        )
    return misfit, f_adj, event_adj, event_config_adj


def adjoint_wavefield(p, event_adj, event_config_adj,
                      model_simulation_name='forward_simulation_homogeneous_model',
                      field='phi_t', iteration=0,
                      sampling_interval_in_time_steps=10):
    simulation_adj = f"adjoint_{iteration}"
    mesh_current   = p.simulations.get_mesh(model_simulation_name)
    p.add_to_project(
        sn.UnstructuredMeshSimulationConfiguration(
            unstructured_mesh=mesh_current,
            name=simulation_adj,
            event_configuration=event_config_adj,
        ),
        overwrite=True,
    )
    p.simulations.launch(
        ranks_per_job=RANKS,
        site_name=SITE_NAME,
        events=event_adj,
        simulation_configuration=simulation_adj,
        extra_output_configuration={
            "volume_data": {
                "sampling_interval_in_time_steps": sampling_interval_in_time_steps,
                "fields": [field],
            },
        },
        delete_conflicting_previous_results=True,
    )
    p.simulations.query(block=True, verbosity=1)
    _, wavefield_adj = extract_data(
        simulation=simulation_adj,
        events=event_adj,
        receiver_field='phi',
        field=field,
    )
    return wavefield_adj


def compute_gradient(forward_wavefield, adjoint_wavefield):
    fwd = forward_wavefield.values.squeeze()
    adj = adjoint_wavefield.values.squeeze()
    t_coord = forward_wavefield.coords.get('t', forward_wavefield.coords.get('time'))
    t_fwd   = t_coord.values
    dt      = float(t_fwd[1] - t_fwd[0])
    if adj.shape[-1] != fwd.shape[-1]:
        t_adj = adjoint_wavefield.coords.get('t', adjoint_wavefield.coords.get('time')).values
        adj   = interp1d(t_adj, adj, axis=-1, kind='linear',
                         bounds_error=False, fill_value=0.0)(t_fwd)
    adj_flipped = np.flip(adj, axis=-1)
    return np.einsum('nt,nt->n', adj_flipped, fwd) * dt


In [ ]:
_xg, _yg = np.linspace(0, 1, 400), np.linspace(0, 1, 400)
_Xg, _Yg = np.meshgrid(_xg, _yg)


def _add_ellipse_overlay(ax, s_locs, r_locs, a, label=True):
    from matplotlib.lines import Line2D
    for s in s_locs:
        for r in r_locs:
            _d1 = np.sqrt((_Xg - s[0])**2 + (_Yg - s[1])**2)
            _d2 = np.sqrt((_Xg - r[0])**2 + (_Yg - r[1])**2)
            ax.contour(_Xg, _Yg, _d1 + _d2, levels=[2 * a],
                       colors='white', linewidths=1.5, linestyles='dashed', alpha=0.85)
    if label:
        ax.scatter([s[0] for s in s_locs], [s[1] for s in s_locs],
                   color='red',  s=60, zorder=5, label='Source')
        ax.scatter([r[0] for r in r_locs], [r[1] for r in r_locs],
                   color='lime', s=60, zorder=5, label='Receiver')
        from matplotlib.lines import Line2D
        ellipse_proxy = Line2D([0], [0], color='white', linewidth=1.5,
                               linestyle='dashed', label='Sensitivity region')
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles=handles + [ellipse_proxy],
                  labels=labels + ['Sensitivity region'],
                  fontsize=8, loc='upper right',
                  facecolor='dimgray', labelcolor='white', edgecolor='white')


def plot_fwi_diagnostics(it, alpha, phi_obs, phi_syn, f_adj, phi_t, phi_t_adj,
                          K_VP, K_roi, x, y, x_roi, y_roi,
                          x0_roi, x1_roi, y0_roi, y1_roi,
                          s_loc, r_loc, a_ell, save=True):
    from matplotlib.gridspec import GridSpec
    from matplotlib.lines import Line2D

    def normalize(arr):
        m = np.nanmax(np.abs(arr))
        return arr / m if m > 0 else arr

    time_wf = phi_obs.time.values

    # Figure 1: waveforms + adjoint source
    fig1, axs = plt.subplots(1, 2, figsize=(14, 4), dpi=100)
    axs[0].plot(time_wf * 1e3, np.array(phi_obs[0]), color='red',   lw=1.5, label=r'$\phi_\mathrm{obs}$')
    axs[0].plot(time_wf * 1e3, np.array(phi_syn[0]), color='green', lw=1.5, label=rf'$\phi^{{({it})}}$')
    axs[0].legend(fontsize=11)
    axs[0].set_xlabel('Time (ms)'); axs[0].set_ylabel(r'$\phi$')
    axs[0].set_title(f'Observed vs synthetic  (iter {it},  alpha={alpha})')
    axs[0].grid(alpha=0.35)
    axs[1].plot(time_wf * 1e3, np.array(-f_adj[0]) * sampling_rate_in_hertz, color='gray', lw=1.5)
    axs[1].axhline(0, color='k', lw=0.5, ls='--')
    axs[1].set_xlabel('Time (ms)'); axs[1].set_ylabel(r'$f^\dagger$')
    axs[1].set_title(r'Adjoint source $f^\dagger$ (time-reversed residual)')
    axs[1].grid(alpha=0.35)
    plt.tight_layout()
    if save:
        plt.savefig(IMAGE_DIR / f'iter_{it:03d}_adjoint_source_alpha_{alpha}_c_{vc}_multiscale.png')
    plt.show()

    # Figure 2: wavefield snapshots
    fwd = phi_t.values.squeeze()
    adj = phi_t_adj.values.squeeze()
    t_fwd = phi_t.coords.get('t', phi_t.coords.get('time')).values
    t_adj_c = phi_t_adj.coords.get('t', phi_t_adj.coords.get('time')).values
    if adj.shape[-1] != fwd.shape[-1]:
        adj = interp1d(t_adj_c, adj, axis=-1, kind='linear',
                       bounds_error=False, fill_value=0.0)(t_fwd)
    adj_flipped = np.flip(adj, axis=-1)
    T = fwd.shape[-1]
    t_indices = [T // 4, T // 2, 3 * T // 4]

    fig2 = plt.figure(figsize=(17, 15), dpi=100)
    gs = GridSpec(3, 4, figure=fig2, width_ratios=[1, 1, 1, 0.04], hspace=0.45, wspace=0.35)
    axes2 = [[fig2.add_subplot(gs[r, c]) for c in range(3)] for r in range(3)]
    cax   = fig2.add_subplot(gs[1, 3])
    col_titles = [r'$\phi_{tt}(x,\,t)$  [normalised]',
                  r'$\phi^\dagger(x,\,T{-}t)$  [normalised]',
                  r'$\phi_{tt}\;\cdot\;\phi^\dagger$  [normalised]']
    for col, title in enumerate(col_titles):
        axes2[0][col].set_title(title, fontsize=12, pad=8)
    tp_last = None
    for row, ti in enumerate(t_indices):
        t_ms  = t_fwd[ti] * 1e3
        s_fwd  = normalize(fwd[:, ti])
        s_adj  = normalize(adj_flipped[:, ti])
        s_prod = normalize(s_fwd * s_adj)
        for col, (snap, lbl) in enumerate([(s_fwd,  r'$\phi_{t}(t)$'),
                                            (s_adj,  r'$\phi^\dagger_{t}(T-t)$'),
                                            (s_prod, r'$\phi_{t}(t)\cdot\phi^\dagger_{t}(T-t)$')]):
            ax = axes2[row][col]
            tp = ax.tripcolor(x, y, snap, cmap='RdBu', shading='gouraud', vmin=-1, vmax=1)
            ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
            ax.set_title(f'{lbl}  at  t = {t_ms:.2f} ms', fontsize=10)
            ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal'); ax.grid(alpha=0.15)
            if row == 1 and col == 2:
                tp_last = tp
    fig2.colorbar(tp_last, cax=cax, label='normalised amplitude')
    if save:
        plt.savefig(IMAGE_DIR / f'iter_{it:03d}_sensitivity_kernel_intermediate_alpha_{alpha}_c_{vc}_multiscale.png')
    plt.show()

    # Figure 3: full sensitivity kernel
    fig3, ax_K = plt.subplots(figsize=(8, 6), dpi=100)
    _K_VP_max = np.max(np.abs(K_VP))
    tp = ax_K.tripcolor(x, y, K_VP, cmap='RdBu', shading='gouraud',
                        vmin=-_K_VP_max, vmax=_K_VP_max)
    plt.colorbar(tp, ax=ax_K, label=r'$\partial J/\partial \ln V_P$')
    _add_ellipse_overlay(ax_K, [s_loc], r_loc, a_ell)
    ax_K.set_xlabel('X (m)'); ax_K.set_ylabel('Y (m)')
    ax_K.set_title(rf'$K_{{VP}}$  —  full domain  (iter {it},  alpha={alpha})', fontsize=12)
    ax_K.set_xlim(0, 1); ax_K.set_ylim(0, 1); ax_K.set_aspect('equal'); ax_K.grid(alpha=0.15)
    plt.tight_layout()
    if save:
        plt.savefig(IMAGE_DIR / f'iter_{it:03d}_sensitivity_kernel_full_alpha_{alpha}_c_{vc}_multiscale.png')
    plt.show()

    # Figure 4: ROI sensitivity kernel
    _d1 = np.sqrt((x_roi - s_loc[0])**2 + (y_roi - s_loc[1])**2)
    _d2 = np.sqrt((x_roi - r_loc[0][0])**2 + (y_roi - r_loc[0][1])**2)
    K_roi_masked = np.where((_d1 + _d2) <= 2 * a_ell, K_roi, 0.0)
    _K_roi_max = np.max(np.abs(K_roi_masked)) if np.max(np.abs(K_roi_masked)) > 0 else 1.0
    fig4, ax_2d = plt.subplots(figsize=(8, 6), dpi=100)
    tp = ax_2d.tripcolor(x_roi, y_roi, K_roi_masked,
                         cmap='RdBu', shading='gouraud', vmin=-_K_roi_max, vmax=_K_roi_max)
    plt.colorbar(tp, ax=ax_2d, label=r'$\partial J/\partial \ln V_P$')
    _add_ellipse_overlay(ax_2d, [s_loc], r_loc, a_ell, label=False)
    _ellipse_proxy = Line2D([0], [0], color='white', linewidth=1.5,
                            linestyle='dashed', label='Sensitivity region')
    ax_2d.legend(handles=[_ellipse_proxy], fontsize=8, loc='upper right',
                 facecolor='dimgray', labelcolor='white', edgecolor='white')
    ax_2d.set_xlabel('X (m)'); ax_2d.set_ylabel('Y (m)')
    ax_2d.set_title(
        f'$K_{{VP}}$ in ROI [{x0_roi},{x1_roi}]x[{y0_roi},{y1_roi}]  (iter {it},  alpha={alpha})',
        fontsize=12,
    )
    ax_2d.set_xlim(x0_roi, x1_roi); ax_2d.set_ylim(y0_roi, y1_roi)
    ax_2d.set_aspect('equal'); ax_2d.grid(alpha=0.15)
    plt.tight_layout()
    if save:
        plt.savefig(IMAGE_DIR / f'iter_{it:03d}_sensitivity_kernel_roi_alpha_{alpha}_c_{vc}_multiscale.png')
    plt.show()


## Multi-scale FWI

Loop over `alpha_list = [0.2, 0.4, 0.6]` (low → high frequency), running
**10 L-BFGS + trust-region iterations** per scale.  The final v(y) profile
recovered at each scale is used to warm-start the next scale.


In [ ]:
# ── Accumulators ---------------------------------------------------------
all_misfit_history = {}   # alpha -> list of misfits
all_model_history  = {}   # alpha -> np.array of model snapshots

warm_vp_nodal = None  # full-mesh nodal VP from previous scale, or None

x0_roi, x1_roi, y0_roi, y1_roi = ROI

for scale_idx, alpha in enumerate(alpha_list):
    print(f"\n{'='*65}")
    print(f"  SCALE {scale_idx+1}/{len(alpha_list)}:  alpha={alpha}  "
          f"f_c={alpha*75e2:.0f} Hz")
    print(f"{'='*65}")

    # ── 1. Per-scale frequency parameters ─────────────────────────────────
    f_c                    = alpha * 75e2
    CENTRAL_FREQUENCY      = 2 * 75e2
    reference_frequency    = CENTRAL_FREQUENCY * 2
    end_time               = end_time_per_alpha[alpha]
    sampling_rate_in_hertz = CENTRAL_FREQUENCY * 200   # global — used by helpers
    time_shift_in_seconds  = 1 / f_c

    wavelet = sn.simple_config.stf.Ricker(
        center_frequency=f_c,
        time_shift_in_seconds=time_shift_in_seconds,
    )
    waveform_config = sn.WaveformSimulationConfiguration(
        start_time_in_seconds=0,
        end_time_in_seconds=end_time,
        time_step_in_seconds=1 / sampling_rate_in_hertz,
    )
    event_config = sn.EventConfiguration(
        wavelet=wavelet,
        waveform_simulation_configuration=waveform_config,
    )

    # ── 2. Project for this scale ─────────────────────────────────────────
    project_name = f'acoustic_forward_single_pairs_alpha_{alpha}'
    p = sn.Project.from_domain(                         # global p used by helpers
        path=Path(PROJECT_DIR, project_name), domain=domain, load_if_exists=True
    )

    # ── 3. Mesh parameters ────────────────────────────────────────────────
    ab_params = salvus.mesh.simple_mesh.basic_mesh.AbsorbingBoundaryParameters(
        free_surface=free_surfaces,
        number_of_wavelengths=number_of_wavelengths,
        reference_velocity=VP,
        reference_frequency=reference_frequency,
    )
    mesh_res = sn.MeshResolution(
        reference_frequency=reference_frequency,
        elements_per_wavelength=elements_per_wavelength,
        model_order=model_order,
    )

    # ── 4. Build meshes ───────────────────────────────────────────────────
    mesh_true = lm.mesh_from_domain(
        domain=domain,
        model=sn.layered_meshing.MeshingProtocol(m_true, ab=ab_params),
        mesh_resolution=mesh_res,
    )
    mesh_homo = lm.mesh_from_domain(
        domain=domain,
        model=sn.layered_meshing.MeshingProtocol(
            sn.material.from_params(rho=RHO, vp=VP), ab=ab_params
        ),
        mesh_resolution=mesh_res,
    )

    # Warm start: assign final VP from previous scale directly onto mesh_homo
    if warm_vp_nodal is not None:
        mesh_homo.attach_field("VP", warm_vp_nodal[mesh_homo.connectivity])
        print(f"  Warm start: VP range = [{warm_vp_nodal.min():.0f}, "
              f"{warm_vp_nodal.max():.0f}] m/s")

    # ── 5. Register events and simulation configs ─────────────────────────
    p.add_to_project(sn.Event(event_name=event_name, sources=source, receivers=receivers))

    obs_sim  = 'forward_simulation_layred_model'
    init_sim = 'forward_simulation_homogeneous_model'

    p.add_to_project(
        sn.UnstructuredMeshSimulationConfiguration(
            unstructured_mesh=mesh_true,
            name=obs_sim,
            event_configuration=event_config,
        ),
        overwrite=True,
    )
    p.add_to_project(
        sn.UnstructuredMeshSimulationConfiguration(
            unstructured_mesh=mesh_homo,
            name=init_sim,
            event_configuration=event_config,
        ),
        overwrite=True,
    )

    # ── 6. Run initial simulations ─────────────────────────────────────────
    t0 = datetime.now()
    forward_simulation(simulation_name=obs_sim,  events=event_name, fields=None)
    forward_simulation(simulation_name=init_sim, events=event_name,
                       fields=['phi_t'],
                       sampling_interval_in_time_steps=SAMPLING_INTERVAL)
    elapsed = (datetime.now() - t0).total_seconds()
    print(f"Initial simulations done in {int(elapsed//60)}m {elapsed%60:.1f}s")

    # ── 7. Extract iteration-0 data ───────────────────────────────────────
    phi_obs, _      = extract_data(obs_sim,  event_name, receiver_field='phi', field=None)
    phi_homo_d, phi_t0 = extract_data(init_sim, event_name, receiver_field='phi', field='phi_t')

    misfit_0, f_adj_0, event_adj_0, event_config_adj_0 = misfit_adjoint_source(
        phi_obs, phi_homo_d, event=event_name, iteration=0,
    )
    print(f"L2 Misfit (iteration 0): {float(misfit_0):.6e}")

    phi_t_adj_0 = adjoint_wavefield(
        p, event_adj_0, event_config_adj_0,
        model_simulation_name=init_sim,
        field='phi_t', iteration=0,
        sampling_interval_in_time_steps=SAMPLING_INTERVAL,
    )

    # ── 8. Mesh/ROI setup ─────────────────────────────────────────────────
    base_mesh = copy.deepcopy(p.simulations.get_mesh(init_sim))
    _x, _y   = base_mesh.points[:, 0], base_mesh.points[:, 1]

    roi_mask = (_x >= x0_roi) & (_x <= x1_roi) & (_y >= y0_roi) & (_y <= y1_roi)
    _x_roi, _y_roi = _x[roi_mask], _y[roi_mask]

    _y_round_full = np.round(_y, decimals=8)
    _y_round_roi  = np.round(_y[roi_mask], decimals=8)

    VP_MIN, VP_MAX = VP * 0.5, VP * 1.5
    _vp_bg_nodal = elemental_nodal_to_nodal_field(
        base_mesh.element_nodal_fields['VP'], base_mesh.connectivity
    ).copy()

    # Fresnel zone / ellipse mask
    time_wf   = phi_homo_d.time.values
    width_3dB = envelope_3dB_width(phi_homo_d[0], time_wf, plot=False)['width'] / 2
    _d_sr_fwi = np.sqrt((s_loc[0] - r_loc[0][0])**2 + (s_loc[1] - r_loc[0][1])**2)
    _a_fwi    = (_d_sr_fwi + VP * width_3dB) / 2

    _ellipse_roi = np.zeros(roi_mask.sum(), dtype=bool)
    for _s in [s_loc]:
        for _r in r_loc:
            _d1 = np.sqrt((_x_roi - _s[0])**2 + (_y_roi - _s[1])**2)
            _d2 = np.sqrt((_x_roi - _r[0])**2 + (_y_roi - _r[1])**2)
            _ellipse_roi |= (_d1 + _d2) <= 2 * _a_fwi
    print(f"Ellipse mask: {_ellipse_roi.sum()} / {roi_mask.sum()} ROI nodes in Fresnel zone")

    # Iteration-0 diagnostics
    K_A_0   = compute_gradient(phi_t0, phi_t_adj_0)
    K_A_0   = np.where(np.isfinite(K_A_0), K_A_0, 0.0)
    vp_n0   = elemental_nodal_to_nodal_field(
        base_mesh.element_nodal_fields['VP'], base_mesh.connectivity
    )
    K_VP_0  = K_A_0 * (-2.0 / vp_n0**2)
    plot_fwi_diagnostics(
        it=0, alpha=alpha,
        phi_obs=phi_obs, phi_syn=phi_homo_d, f_adj=f_adj_0,
        phi_t=phi_t0, phi_t_adj=phi_t_adj_0,
        K_VP=K_VP_0, K_roi=K_VP_0[roi_mask],
        x=_x, y=_y, x_roi=_x_roi, y_roi=_y_roi,
        x0_roi=x0_roi, x1_roi=x1_roi, y0_roi=y0_roi, y1_roi=y1_roi,
        s_loc=s_loc, r_loc=r_loc, a_ell=_a_fwi,
    )

    # ── 9. Per-scale closures ─────────────────────────────────────────────
    _fwi_iter   = [0]
    _fwi_ls     = [0]
    _diag_cache = {}

    _event_config_fwd = sn.EventConfiguration(
        wavelet=wavelet,
        waveform_simulation_configuration=waveform_config,
    )

    def _average_vp_in_x(vp_nodal,
                          _y_round=_y_round_full,
                          _y_unique=np.unique(np.round(_y, decimals=8))):
        out = vp_nodal.copy()
        for yv in _y_unique:
            mask = _y_round == yv
            out[mask] = vp_nodal[mask].mean()
        return out

    def objective(m_flat,
                  _scale=scale_idx, _alpha=alpha,
                  _roi=roi_mask, _bg=_vp_bg_nodal,
                  _ell=_ellipse_roi, _bm=base_mesh,
                  _vmin=VP_MIN, _vmax=VP_MAX,
                  _phi_obs=phi_obs, _cfg=_event_config_fwd,
                  _si=SAMPLING_INTERVAL, _en=event_name):
        it   = _fwi_iter[0]
        ls   = _fwi_ls[0]
        _fwi_ls[0] += 1
        tag  = f'fwi_s{_scale}_{it:03d}_g' if ls == 0 else f'fwi_s{_scale}_{it:03d}_ls{ls}'

        vp_roi   = np.clip(np.exp(m_flat), _vmin, _vmax)
        vp_nodal = _bg.copy()
        vp_nodal[_roi] = vp_roi
        vp_nodal = _average_vp_in_x(vp_nodal)
        vp_roi   = vp_nodal[_roi]

        mesh_it = copy.deepcopy(_bm)
        mesh_it.attach_field("VP", vp_nodal[mesh_it.connectivity])
        p.add_to_project(
            sn.UnstructuredMeshSimulationConfiguration(
                unstructured_mesh=mesh_it,
                name=tag,
                event_configuration=_cfg,
            ),
            overwrite=True,
        )
        forward_simulation(tag, _en, fields=['phi_t'],
                           sampling_interval_in_time_steps=_si)
        phi_syn, phi_t_it = extract_data(tag, _en, receiver_field='phi', field='phi_t')

        misfit, f_adj_wf, event_adj, event_config_adj = misfit_adjoint_source(
            _phi_obs, phi_syn, event=_en, iteration=tag,
        )
        phi_adj_it = adjoint_wavefield(
            p, event_adj, event_config_adj,
            model_simulation_name=tag,
            field='phi_t', iteration=tag,
            sampling_interval_in_time_steps=_si,
        )
        K_A    = compute_gradient(phi_t_it, phi_adj_it)
        K_A    = np.where(np.isfinite(K_A), K_A, 0.0)
        K_lnVP = K_A * (-2.0 / vp_nodal**2)

        _diag_cache.update({
            'phi_syn': phi_syn, 'f_adj': f_adj_wf,
            'phi_t': phi_t_it,  'phi_t_adj': phi_adj_it,
            'K_VP':  K_lnVP,    'K_roi': K_lnVP[_roi],
        })
        grad_lnvp = K_lnVP[_roi].copy()
        grad_lnvp[~_ell] = 0.0
        label = 'grad' if ls == 0 else f'ls_{ls}'
        print(f"  s{_scale+1} iter {it:3d} {label:6s} | "
              f"misfit = {float(misfit):.6e} | "
              f"|g|_max = {np.max(np.abs(grad_lnvp)):.3e}")
        return float(misfit), grad_lnvp.ravel()

    def _mesh_vp_roi(vp_roi_vals,
                     _roi=roi_mask, _bg=_vp_bg_nodal,
                     _vmin=VP_MIN, _vmax=VP_MAX):
        vp_n = _bg.copy()
        vp_n[_roi] = vp_roi_vals
        vp_n = _average_vp_in_x(vp_n)
        return np.clip(vp_n[_roi], _vmin, _vmax)

    # ── 10. Initialise m for this scale ───────────────────────────────────
    if warm_vp_nodal is not None:
        m = np.log(np.clip(warm_vp_nodal[roi_mask], VP_MIN, VP_MAX))
    else:
        m = np.full(roi_mask.sum(), np.log(VP))

    # ── 11. L-BFGS + trust region (10 accepted iterations) ────────────────
    delta_tr   = 0.3
    delta_max  = 1.0
    eta1, eta2 = 0.0001, 0.75
    gamma1, gamma2 = 0.5, 0.9
    sigma      = 2.0
    M_lbfgs    = 10
    MAX_TRIES  = MAX_ITER_PER_SCALE * 3

    misfit_history = []
    model_history  = [_mesh_vp_roi(np.clip(np.exp(m), VP_MIN, VP_MAX))]
    s_list, y_list, rho_list = [], [], []
    prev_grad     = None
    prev_s        = None
    cached_misfit = None
    cached_grad   = None

    _fwi_iter[0] = 0
    _fwi_ls[0]   = 0

    k = 0       # accepted iterations
    tries = 0   # total attempts

    while k < MAX_ITER_PER_SCALE and tries < MAX_TRIES:
        tries += 1
        _fwi_iter[0] = k
        _fwi_ls[0]   = 0

        if cached_misfit is not None:
            misfit, grad = cached_misfit, cached_grad
            print("  [cache] reusing gradient from rejected step")
        else:
            misfit, grad = objective(m)

        # L-BFGS memory update
        if prev_grad is not None and prev_s is not None:
            y_k = grad - prev_grad
            sy  = np.dot(prev_s, y_k)
            if sy > 0:
                s_list.append(prev_s.copy())
                y_list.append(y_k.copy())
                rho_list.append(1.0 / sy)
                if len(s_list) > M_lbfgs:
                    s_list.pop(0); y_list.pop(0); rho_list.pop(0)

        # L-BFGS two-loop
        q, alpha_lbfgs = grad.copy(), []
        for _s, _y_v, _rho in zip(reversed(s_list), reversed(y_list), reversed(rho_list)):
            a = _rho * np.dot(_s, q); q -= a * _y_v; alpha_lbfgs.append(a)
        gamma_h = (np.dot(s_list[-1], y_list[-1]) / np.dot(y_list[-1], y_list[-1])
                   if s_list else 1.0)
        r = gamma_h * q
        for _s, _y_v, _rho, a in zip(s_list, y_list, rho_list, reversed(alpha_lbfgs)):
            r += _s * (a - _rho * np.dot(_y_v, r))
        d = -r

        d_max = np.abs(d).max()
        if d_max < 1e-30:
            print(f"  iter {k} | direction vanished — stopping scale"); break
        step        = delta_tr * (d / d_max)
        m_new       = np.clip(m + step, np.log(VP_MIN), np.log(VP_MAX))
        actual_step = m_new - m

        _fwi_ls[0] = 1
        misfit_new, grad_new = objective(m_new)

        actual_red    = misfit - misfit_new
        predicted_red = -np.dot(grad, actual_step)
        rho_k   = actual_red / predicted_red if abs(predicted_red) > 1e-30 else 0.0
        step_norm = np.abs(actual_step).max()

        if rho_k >= eta1:
            prev_grad = grad.copy()
            _vp_acc   = np.clip(np.exp(m_new), VP_MIN, VP_MAX)
            for _yv in np.unique(_y_round_roi):
                _ym = _y_round_roi == _yv
                _vp_acc[_ym] = _vp_acc[_ym].mean()
            m_proj = np.clip(np.log(_vp_acc), np.log(VP_MIN), np.log(VP_MAX))
            prev_s        = m_proj - m
            m             = m_proj
            cached_misfit = misfit_new
            cached_grad   = grad_new
            misfit_history.append(misfit_new)
            model_history.append(_mesh_vp_roi(_vp_acc))
            status = "accepted"
            k += 1

            plot_fwi_diagnostics(
                it=k, alpha=alpha,
                phi_obs=phi_obs,
                phi_syn=_diag_cache['phi_syn'],
                f_adj=_diag_cache['f_adj'],
                phi_t=_diag_cache['phi_t'],
                phi_t_adj=_diag_cache['phi_t_adj'],
                K_VP=_diag_cache['K_VP'],
                K_roi=_diag_cache['K_roi'],
                x=_x, y=_y, x_roi=_x_roi, y_roi=_y_roi,
                x0_roi=x0_roi, x1_roi=x1_roi, y0_roi=y0_roi, y1_roi=y1_roi,
                s_loc=s_loc, r_loc=r_loc, a_ell=_a_fwi,
            )
        else:
            cached_misfit = misfit
            cached_grad   = grad
            status = "REJECTED"

        # Trust region radius update
        if rho_k < eta1:
            delta_tr = min(delta_tr / sigma, step_norm)
        elif rho_k > eta2 and step_norm >= gamma2 * delta_tr:
            delta_tr = min(sigma * delta_tr, delta_max)
        elif step_norm < gamma1 * delta_tr:
            delta_tr = max(delta_tr / sigma, sigma * step_norm)

        vp_mean   = _mesh_vp_roi(np.clip(np.exp(m), VP_MIN, VP_MAX))[_ellipse_roi].mean()
        grad_norm = np.linalg.norm(grad)
        print(f"  Scale {scale_idx+1} k={k:2d}  VP_mean={vp_mean:.1f} m/s  "
              f"chi={misfit:.6e}  ||g||={grad_norm:.3e}  "
              f"||Δm||={step_norm:.4f}  Δtr={delta_tr:.4f}  ({status})")

    model_history = np.array(model_history)

    # Convergence plot for this scale
    fig_cv, ax_cv = plt.subplots(figsize=(6, 4), dpi=100)
    ax_cv.semilogy(misfit_history, marker='o', linewidth=1.5)
    ax_cv.set_xlabel('Accepted iteration')
    ax_cv.set_ylabel('L2 Misfit')
    ax_cv.set_title(f'Scale {scale_idx+1}  alpha={alpha}  f_c={f_c:.0f} Hz')
    ax_cv.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    all_misfit_history[alpha] = misfit_history
    all_model_history[alpha]  = model_history

    # ── 12. Build full-mesh VP for warm-starting next scale ───────────────
    warm_vp_nodal = _vp_bg_nodal.copy()
    warm_vp_nodal[roi_mask] = model_history[-1]
    warm_vp_nodal = _average_vp_in_x(warm_vp_nodal)
    print(f"  Warm start VP: range=[{warm_vp_nodal.min():.0f}, "
          f"{warm_vp_nodal.max():.0f}] m/s")

print("\nMulti-scale FWI complete.")

# ── Save combined results ────────────────────────────────────────────────
np.save(DATA_DIR / "misfit_history_multiscale.npy", all_misfit_history, allow_pickle=True)
np.save(DATA_DIR / "model_history_multiscale.npy",  all_model_history,  allow_pickle=True)
print(f"Saved misfit_history_multiscale and model_history_multiscale to {DATA_DIR}")


## Visualise Recovered Model

In [ ]:
# ── Combined misfit convergence across all scales ─────────────────────────
fig, ax = plt.subplots(figsize=(8, 4), dpi=100)
offset = 0
for alpha_val, mh in all_misfit_history.items():
    iters = np.arange(offset, offset + len(mh))
    ax.semilogy(iters, mh, marker='o', lw=1.5, label=f'alpha={alpha_val}')
    if mh:
        ax.axvline(offset + len(mh) - 1, color='gray', lw=0.8, ls='--')
    offset += len(mh)
ax.set_xlabel('Cumulative accepted iteration')
ax.set_ylabel('L2 Misfit')
ax.set_title('Multi-scale FWI convergence')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
all_model_history[0.2][-1].mean()

In [ ]:
# ── Intermediate model at iteration 10 (end of scale 1, alpha=0.2) ────────
_alpha = 0.2 
_vp_roi_10 = all_model_history[0.2][-1]               

# Reconstruct full-mesh VP: homogeneous background outside ROI
_vp_full_10 = np.full(len(_x), VP)
_vp_full_10[roi_mask] = _vp_roi_10

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=100)

# ── Left: 2-D VP field ───────────────────────────────────────────────────
_vmin10 = _vp_full_10[roi_mask].min() - 20
_vmax10 = _vp_full_10[roi_mask].max() + 20
tp = axes[0].tripcolor(
    _x, _y, _vp_full_10,
    cmap='RdBu_r', shading='gouraud',
    vmin=_vmin10, vmax=_vmax10,
)
plt.colorbar(tp, ax=axes[0], label='VP  [m/s]')
from matplotlib.patches import Rectangle
x0_roi, x1_roi, y0_roi, y1_roi = ROI
axes[0].add_patch(Rectangle(
    (x0_roi, y0_roi), x1_roi - x0_roi, y1_roi - y0_roi,
    linewidth=1.5, edgecolor='black', facecolor='none', linestyle='--',
))
axes[0].scatter([s_loc[0]], [s_loc[1]], color='red',  s=60, zorder=5, label='Source')
axes[0].scatter([r[0] for r in r_loc], [r[1] for r in r_loc],
                color='lime', s=60, zorder=5, label='Receiver')
axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)')
axes[0].set_title(f'VP at iteration 10  (alpha={_alpha})', fontsize=12)
axes[0].set_xlim(x0_dom, x1_dom); axes[0].set_ylim(y0_dom, y1_dom)
axes[0].set_aspect('equal'); axes[0].grid(alpha=0.15)
axes[0].legend(fontsize=8, loc='upper right')

# ── Right: v(y) profile ──────────────────────────────────────────────────
_y_unique   = np.unique(_y_round_full)
_vp_rec_10  = np.array([_vp_full_10[_y_round_full == yv].mean() for yv in _y_unique])
_vp_true    = np.where((_y_unique >= y0_roi) & (_y_unique <= y1_roi), vc, VP)

axes[1].plot(np.full_like(_y_unique, VP), _y_unique,
             color='gray',      lw=1.5, ls='--', label=f'Initial (VP={VP:.0f} m/s)')
axes[1].plot(_vp_rec_10, _y_unique, color='steelblue', lw=2,    label='Recovered (iter 10)')
axes[1].plot(_vp_true,   _y_unique, color='red',       lw=1.5, ls=':', label=f'True (vc={vc} m/s)')
axes[1].axhspan(y0_roi, y1_roi, alpha=0.08, color='green', label='ROI')
axes[1].set_xlabel('VP  [m/s]'); axes[1].set_ylabel('Y (m)')
axes[1].set_title('v(y) profile — iteration 10', fontsize=12)
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
axes[1].set_ylim(y0_dom, y1_dom)

plt.suptitle(
    f'Multi-scale FWI — intermediate model at iteration 10  (alpha={_alpha})',
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.savefig(IMAGE_DIR / f'intermediate_model_iter10_alpha{_alpha}_multiscale.png')
plt.show()

print(f"Iter-10 VP in ROI: mean={_vp_roi_10.mean():.1f} m/s  "
      f"(true={vc} m/s,  error={abs(_vp_roi_10.mean()-vc):.1f} m/s)")


In [ ]:
# ── Final recovered model — 2-D VP field and v(y) profile ────────────────
# Uses warm_vp_nodal (full-mesh nodal VP) set at end of the last scale,
# and _x, _y, _y_round_full from the same scale's mesh.

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=100)

# ── Left: 2-D VP field ────────────────────────────────────────────────────
tp = axes[0].tripcolor(
    _x, _y, warm_vp_nodal,
    cmap='RdBu_r', shading='gouraud',
    vmin=VP * 0.85, vmax=VP * 1.05,
)
plt.colorbar(tp, ax=axes[0], label='VP  [m/s]')
# ROI boundary
x0_roi, x1_roi, y0_roi, y1_roi = ROI
from matplotlib.patches import Rectangle
axes[0].add_patch(Rectangle(
    (x0_roi, y0_roi), x1_roi - x0_roi, y1_roi - y0_roi,
    linewidth=1.5, edgecolor='black', facecolor='none', linestyle='--',
))
axes[0].scatter([s_loc[0]], [s_loc[1]], color='red',  s=60, zorder=5, label='Source')
axes[0].scatter([r[0] for r in r_loc], [r[1] for r in r_loc],
                color='lime', s=60, zorder=5, label='Receiver')
axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)')
axes[0].set_title(f'Final recovered VP  (alpha={alpha_list[-1]})', fontsize=12)
axes[0].set_xlim(x0_dom, x1_dom); axes[0].set_ylim(y0_dom, y1_dom)
axes[0].set_aspect('equal'); axes[0].grid(alpha=0.15)
axes[0].legend(fontsize=8, loc='upper right')

# ── Right: v(y) profile ───────────────────────────────────────────────────
_y_unique  = np.unique(_y_round_full)
_vp_rec    = np.array([warm_vp_nodal[_y_round_full == yv].mean() for yv in _y_unique])

# True model: vc inside ROI y-range, VP elsewhere
_vp_true = np.where((_y_unique >= y0_roi) & (_y_unique <= y1_roi), vc, VP)

axes[1].plot(np.full_like(_y_unique, VP), _y_unique,
             color='gray', lw=1.5, ls='--', label=f'Initial (VP={VP:.0f} m/s)')
axes[1].plot(_vp_rec,  _y_unique, color='steelblue', lw=2,   label='Recovered')
axes[1].plot(_vp_true, _y_unique, color='red',        lw=1.5, ls=':', label=f'True (vc={vc} m/s)')
axes[1].axhspan(y0_roi, y1_roi, alpha=0.08, color='green', label='ROI')
axes[1].set_xlabel('VP  [m/s]'); axes[1].set_ylabel('Y (m)')
axes[1].set_title('v(y) profile — initial vs recovered vs true', fontsize=12)
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
axes[1].set_ylim(y0_dom, y1_dom)

plt.suptitle(
    f'Multi-scale FWI — final model  '
    f'(alpha = {alpha_list})',
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'final_model_multiscale.png')
plt.show()

print(f"Recovered VP in ROI: mean={warm_vp_nodal[roi_mask].mean():.1f} m/s  "
      f"(true={vc} m/s,  error={abs(warm_vp_nodal[roi_mask].mean()-vc):.1f} m/s)")


In [ ]:
# ── RMSE over iterations (all scales combined) ────────────────────────────
rmse_all = []
for mh in all_model_history.values():
    for k in range(len(mh)):
        rmse_all.append(np.sqrt(np.mean((mh[k] - vc) ** 2)))

fig, ax = plt.subplots(figsize=(8, 4), dpi=100)
ax.plot(rmse_all, 'o-', color='steelblue', lw=2, ms=5)
ax.set_xlabel('Cumulative accepted iteration')
ax.set_ylabel('RMSE in ROI  [m/s]')
ax.set_title(f'RMSE vs iteration — multi-scale FWI  (true VP = {vc} m/s)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'rmse_iterations_multiscale.png')
plt.show()


In [ ]:
rmse_all